# 04d -- Draft-Pick Value Curve (KTC RDP + DraftSharks)

**Purpose:** Scrape generic (year, round[, tier]) pick-value curves from KTC RDP
and DraftSharks (dynasty TE-premium superflex) into a small reference table.
Neither source resolves to a specific `dim_draft_pick` row -- both publish
market value **by bucket** (KTC: Early/Mid/Late thirds of a round; DraftSharks:
one flat value per future round, no split). Forcing these into the
player-identity `fact_dynasty_ranking_metrics` EAV would muddy its grain, so
they get their own small transformer-style table instead (like `dim_position`/
`dim_school`) -- resolving a *real* pick to a curve value, and interpolating
our 28-pick rounds onto these sources' 12-team grid, is a Slice 2 (backend)
concern, not this notebook's.

**Output:** `data/dim_pick_value_curve.parquet`, grain
`(snapshot_date, source_name, draft_year, round, tier)`. Replace-by-partition
on `(snapshot_date, source_name)` via `etl.load_replace_partition` -- same
idempotent pattern `04b` uses. Run with CWD = repo root.

In [1]:
# ---- Setup & Config ---------------------------------------------------------
import json
import re
import time
from datetime import date
from pathlib import Path

import pandas as pd

import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG, _make_session

TABLE = "dim_pick_value_curve"
SNAPSHOT_DATE = date.today().isoformat()
SNAP_TS = pd.to_datetime(SNAPSHOT_DATE)
print(f"snapshot_date={SNAPSHOT_DATE}")

snapshot_date=2026-07-17


In [2]:
# ---- KTC RDP scrape -----------------------------------------------------
# Same page/parse shape as 04b's KTCDynastyScraper (var playersArray = [...]),
# but targeting the RDP-filtered view and keeping (not skipping) RDP rows.
# RDP playerName is a bucket label: "{year} {Early|Mid|Late} {round}{ordinal}".
KTC_RDP_URL = "https://keeptradecut.com/dynasty-rankings?filters=RDP"
KTC_PLAYERS_PAT = re.compile(r"var playersArray\s*=\s*(\[.*?\]);", re.DOTALL)
KTC_RDP_NAME_PAT = re.compile(r"^(\d{4})\s+(Early|Mid|Late)\s+(\d+)(?:st|nd|rd|th)$")
KTC_HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

def fetch_ktc_rdp_curve(delay: float = 2.0) -> pd.DataFrame:
    session = _make_session()
    session.headers.update(KTC_HEADERS)
    time.sleep(delay)
    resp = session.get(KTC_RDP_URL, timeout=30)
    resp.raise_for_status()
    m = KTC_PLAYERS_PAT.search(resp.text)
    if not m:
        raise ValueError("var playersArray not found in KTC RDP HTML")
    players = json.loads(m.group(1))

    raw_path = Path(CFG.data_dir) / "raw" / f"ktc_rdp_{SNAPSHOT_DATE}.json"
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(json.dumps(players, indent=2), encoding="utf-8")

    rows = []
    for p in players:
        if p.get("position") != "RDP":
            continue
        nm = KTC_RDP_NAME_PAT.match(str(p.get("playerName", "")).strip())
        if not nm:
            print(f"  WARN: unparsed RDP name {p.get('playerName')!r} -- skipped")
            continue
        year, tier, rnd = nm.groups()
        value = p.get("superflexValues", {}).get("value")
        if value is None:
            continue
        rows.append({
            "snapshot_date": SNAP_TS, "source_name": "KTC",
            "draft_year": int(year), "round": int(rnd), "tier": tier,
            "value": float(value),
        })
    return pd.DataFrame(rows), raw_path

ktc_curve, ktc_raw_path = fetch_ktc_rdp_curve()
print(f"[ok] KTC RDP: {len(ktc_curve)} bucket rows -> {ktc_raw_path}")

[ok] KTC RDP: 36 bucket rows -> C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\data\raw\ktc_rdp_2026-07-17.json


In [3]:
# ---- DraftSharks (dynasty TE-premium superflex) scrape -------------------
# Data ships inline as `var vueAppData = {...}` (a Vue SSR payload, no
# separate API call). `pickDefaultScorings` mixes this year's *already-drafted*
# rookie-slot rows (player_id set) with future-year round buckets
# (player_id: null) -- only the latter are pick-value curve rows. Future years
# get ONE flat value per round (no Early/Mid/Late split) on DraftSharks'
# internal 12-team grid (`num_teams` on the matched scoring config) -> tier="All".
DRAFTSHARKS_URL = "https://www.draftsharks.com/trade-value-chart/dynasty/te-premium-superflex"
DRAFTSHARKS_CONFIG_ID = "dynastyTePremiumSuperflex"  # matches the URL's te-premium-superflex slug, dynasty view

def _extract_vue_app_data(html: str) -> dict:
    marker = "var vueAppData = "
    start = html.find(marker)
    if start == -1:
        raise ValueError("var vueAppData not found in DraftSharks HTML")
    start += len(marker)
    depth = 0
    for i, ch in enumerate(html[start:]):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(html[start:start + i + 1])
    raise ValueError("unbalanced braces parsing vueAppData")

def fetch_draftsharks_curve(delay: float = 2.0) -> pd.DataFrame:
    session = _make_session()
    session.headers.update(KTC_HEADERS)  # generic browser UA works for both sites
    time.sleep(delay)
    resp = session.get(DRAFTSHARKS_URL, timeout=30)
    resp.raise_for_status()
    data = _extract_vue_app_data(resp.text)

    raw_path = Path(CFG.data_dir) / "raw" / f"draftsharks_pickvalue_{SNAPSHOT_DATE}.json"
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(json.dumps(data, indent=2), encoding="utf-8")

    cfg = next((c for c in data["scoringConfigurations"] if c["id"] == DRAFTSHARKS_CONFIG_ID), None)
    if cfg is None:
        raise ValueError(f"scoring config {DRAFTSHARKS_CONFIG_ID!r} not found")
    num_teams = cfg["num_teams"]
    mvp_id = cfg["mvpBoardScoringId"]

    rows = []
    for p in data["pickDefaultScorings"]:
        if p["mvp_board_scoring_id"] != mvp_id or p["player_id"] is not None:
            continue  # not our config, or an already-drafted current-year slot
        year_s, seq_s = p["pick_id"].split("-")
        seq = int(seq_s)
        rnd = (seq - 1) // num_teams + 1
        rows.append({
            "snapshot_date": SNAP_TS, "source_name": "DraftSharks",
            "draft_year": int(year_s), "round": rnd, "tier": "All",
            "value": float(p["ds_value"]),
        })
    # collapse the per-pick rows (all equal within a round, since DraftSharks has no split)
    df = pd.DataFrame(rows).drop_duplicates(subset=["snapshot_date", "source_name", "draft_year", "round", "tier"])
    return df, raw_path, num_teams

ds_curve, ds_raw_path, ds_num_teams = fetch_draftsharks_curve()
print(f"[ok] DraftSharks: {len(ds_curve)} bucket rows (num_teams={ds_num_teams}) -> {ds_raw_path}")

[ok] DraftSharks: 10 bucket rows (num_teams=12) -> C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\data\raw\draftsharks_pickvalue_2026-07-17.json


In [4]:
# ---- Combine + load (replace-by-(snapshot_date, source_name)) -----------
curve = pd.concat([ktc_curve, ds_curve], ignore_index=True)
assert curve.duplicated(subset=["snapshot_date", "source_name", "draft_year", "round", "tier"]).sum() == 0, \
    "grain violated: duplicate (snapshot_date, source_name, draft_year, round, tier)"

n = etl.load_replace_partition(curve, CFG.path(TABLE))
print(f"[ok] {len(curve)} curve rows this run (table now {n} total) -> {CFG.path(TABLE)}")

[ok] 46 curve rows this run (table now 46 total) -> C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\data/dim_pick_value_curve.parquet


In [5]:
# ---- Validation -----------------------------------------------------------
pv = pd.read_parquet(CFG.path(TABLE))
print(f"total rows: {len(pv)} | sources: {pv['source_name'].unique().tolist()} "
      f"| draft_years: {sorted(pv['draft_year'].unique().tolist())}")
print()
print(pv.sort_values(["source_name", "draft_year", "round", "tier"])
      .to_string(index=False))

total rows: 46 | sources: ['KTC', 'DraftSharks'] | draft_years: [2026, 2027, 2028]

snapshot_date source_name  draft_year  round  tier  value
   2026-07-17 DraftSharks        2027      1   All   30.3
   2026-07-17 DraftSharks        2027      2   All   14.2
   2026-07-17 DraftSharks        2027      3   All    9.5
   2026-07-17 DraftSharks        2027      4   All    5.9
   2026-07-17 DraftSharks        2027      5   All    5.0
   2026-07-17 DraftSharks        2028      1   All   28.2
   2026-07-17 DraftSharks        2028      2   All   13.2
   2026-07-17 DraftSharks        2028      3   All    8.8
   2026-07-17 DraftSharks        2028      4   All    5.5
   2026-07-17 DraftSharks        2028      5   All    4.6
   2026-07-17         KTC        2026      1 Early 5604.0
   2026-07-17         KTC        2026      1  Late 3917.0
   2026-07-17         KTC        2026      1   Mid 4594.0
   2026-07-17         KTC        2026      2 Early 3260.0
   2026-07-17         KTC        2026      2  